In [1]:
%pip install torch prophet lightgbm wandb kaggle -Uq

import torch
print("PyTorch:", torch.__version__)

Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.13.0


In [2]:
import os
import getpass

import wandb

_wandb_api_key_env = os.environ.get("WANDB_API_KEY")
os.environ.pop("WANDB_API_KEY", None)
try:
    wandb.logout()
except Exception:
    pass

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"

wandb_key = _wandb_api_key_env or getpass.getpass("Paste your W&B API key: ").strip()
wandb.login(key=wandb_key, relogin=True, force=True, verify=True)

who = wandb.Api().viewer
print("Logged in as:", who.username, "| entity:", who.entity)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/macbookpro/.netrc
wandb: Currently logged in as: toberi23 (toberi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in as: toberi23 | entity: toberi23-free-university-of-tbilisi-


In [3]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import pickle
import logging
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

MODEL_TAG = "Ensemble_DLinear_LightGBM_Prophet"
WANDB_GROUP = "Ensemble_DLinear_LightGBM_Prophet_Inference"

DLINEAR_ARTIFACT_NAME = "dlinear_forecast_pipeline"
LIGHTGBM_ARTIFACT_NAME = "lightgbm_forecast_pipeline"
PROPHET_ARTIFACT_NAME = "prophet_forecast_pipeline"

DLINEAR_PKL_FILENAME = "dlinear_pipeline.pkl"
LIGHTGBM_PKL_FILENAME = "lightgbm_pipeline.pkl"
PROPHET_PKL_FILENAME = "prophet_pipeline.pkl"

SUBMISSION_FILENAME = "submission_Ensemble_DLinear_LightGBM_Prophet.csv"

pd.set_option("display.width", 200)

In [4]:
import sys
import subprocess
import zipfile

_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
_KAGGLE_MOUNT_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
_IN_COLAB = "google.colab" in sys.modules

def _download_via_kaggle_cli(target_dir):
    os.makedirs(target_dir, exist_ok=True)
    has_token = os.environ.get("KAGGLE_API_TOKEN") or os.path.exists(os.path.expanduser("~/.kaggle/access_token"))
    if not has_token:
        token = getpass.getpass(
            "Paste your Kaggle API token (kaggle.com -> profile picture -> Settings -> API -> "
            "Create New Token): "
        ).strip()
        os.environ["KAGGLE_API_TOKEN"] = token

    subprocess.run(
        ["kaggle", "competitions", "download", "-c", "walmart-recruiting-store-sales-forecasting",
         "-p", target_dir],
        check=True,
    )
    zip_path = os.path.join(target_dir, "walmart-recruiting-store-sales-forecasting.zip")
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target_dir)
        os.remove(zip_path)

COMP = os.environ.get("WALMART_DATA_DIR")
if not COMP:
    if os.path.isdir(_LOCAL_DATA_DIR):
        COMP = _LOCAL_DATA_DIR
    elif os.path.isdir(_KAGGLE_MOUNT_DIR):
        COMP = _KAGGLE_MOUNT_DIR
    elif _IN_COLAB:
        print("Running on Colab with no local data found -- downloading via the Kaggle API...")
        _download_via_kaggle_cli(_LOCAL_DATA_DIR)
        COMP = _LOCAL_DATA_DIR
    else:
        COMP = _KAGGLE_MOUNT_DIR
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be \'train\' or \'test\'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_test = load_merged("test")
print("Test:", df_test.shape, "|", df_test["Store"].nunique(), "stores,",
      df_test[["Store", "Dept"]].drop_duplicates().shape[0], "(Store, Dept) series")
df_test.head()

Reading competition data from: data/walmart-recruiting-store-sales-forecasting
Test: (115064, 15) | 45 stores, 3169 (Store, Dept) series


,Store,Dept,Date,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.70,50.82,3639.90,2737.42,223.462779,6.573
1,1,1,2012-11-09,False,A,151315,61.24,3.314,11421.32,3370.89,40.28,4646.79,6154.16,223.481307,6.573
2,1,1,2012-11-16,False,A,151315,52.92,3.252,9696.28,292.10,103.78,1133.15,6612.69,223.512911,6.573
3,1,1,2012-11-23,True,A,151315,56.23,3.211,883.59,4.17,74910.32,209.91,303.32,223.561947,6.573
4,1,1,2012-11-30,False,A,151315,52.34,3.207,2460.03,NaN,3838.35,150.57,6966.34,223.610984,6.573


In [5]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name=f"{MODEL_TAG}_Inference", job_type="inference",
                  tags=[MODEL_TAG, "DLinear", "LightGBM", "Prophet", "inference"])

In [6]:
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin

SEED = 42
torch.manual_seed(SEED)
torch.set_num_threads(1)

HORIZON_WEEKS = 39

class MovingAverage(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size - 1 - pad, 1)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.permute(0, 2, 1)).permute(0, 2, 1)

class SeriesDecomposition(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAverage(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        return x - trend, trend

class DLinear(nn.Module):

    def __init__(self, lookback, horizon, n_channels, kernel_size=7, individual=True, normalize=True):
        super().__init__()
        assert kernel_size < lookback, "kernel_size must be smaller than the lookback window"
        self.decomp = SeriesDecomposition(kernel_size)
        self.individual = individual
        self.normalize = normalize
        self.n_channels = n_channels
        self.horizon = horizon

        if individual:
            self.seasonal_weight = nn.Parameter(torch.empty(n_channels, horizon, lookback))
            self.trend_weight = nn.Parameter(torch.empty(n_channels, horizon, lookback))
            self.seasonal_bias = nn.Parameter(torch.zeros(n_channels, horizon))
            self.trend_bias = nn.Parameter(torch.zeros(n_channels, horizon))
            for w in (self.seasonal_weight, self.trend_weight):
                nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        else:
            self.seasonal_linear = nn.Linear(lookback, horizon)
            self.trend_linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        if self.normalize:
            mean = x.mean(dim=1, keepdim=True)
            std = x.std(dim=1, keepdim=True) + 1e-5
            x = (x - mean) / std

        seasonal, trend = self.decomp(x)
        seasonal = seasonal.permute(0, 2, 1)
        trend = trend.permute(0, 2, 1)

        if self.individual:
            seasonal_out = torch.einsum("bcl,chl->bch", seasonal, self.seasonal_weight) + self.seasonal_bias
            trend_out = torch.einsum("bcl,chl->bch", trend, self.trend_weight) + self.trend_bias
        else:
            seasonal_out = self.seasonal_linear(seasonal)
            trend_out = self.trend_linear(trend)

        out = (seasonal_out + trend_out).permute(0, 2, 1)
        if self.normalize:
            out = out * std + mean
        return out

def build_wide_matrix(df: pd.DataFrame) -> pd.DataFrame:
    key = df["Store"].astype(str) + "|" + df["Dept"].astype(str)
    wide = df.assign(_key=key).pivot_table(index="Date", columns="_key", values="Weekly_Sales")

    full_index = pd.date_range(wide.index.min(), wide.index.max(), freq="7D")
    wide = wide.reindex(full_index)

    filled = wide.ffill()
    filled = filled.fillna(filled.mean())
    filled = filled.fillna(0.0)
    return filled

def make_windows(wide: pd.DataFrame, lookback: int, horizon: int = HORIZON_WEEKS):
    arr = wide.to_numpy(dtype=np.float32)
    T = arr.shape[0]
    n_windows = T - lookback - horizon + 1
    if n_windows < 1:
        raise ValueError(
            f"Not enough history ({T} weeks) for lookback={lookback} + horizon={horizon}."
        )
    Xs = np.stack([arr[t:t + lookback] for t in range(n_windows)])
    Ys = np.stack([arr[t + lookback:t + lookback + horizon] for t in range(n_windows)])
    return Xs, Ys, n_windows

CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

class DLinearForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, lookback_weeks=39, kernel_size=7, individual=True, normalize=True,
                 learning_rate=0.01, weight_decay=0.0, epochs=60, batch_size=32,
                 horizon_weeks=HORIZON_WEEKS, use_sample_weight=False,
                 use_christmas_shift=False, christmas_bulge_threshold=1.10, device=None):
        self.lookback_weeks = lookback_weeks
        self.kernel_size = kernel_size
        self.individual = individual
        self.normalize = normalize
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.horizon_weeks = horizon_weeks
        self.use_sample_weight = use_sample_weight
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold
        self.device = device

    def _resolve_device(self):
        return self.device or ("cuda" if torch.cuda.is_available() else "cpu")

    def fit(self, X, y=None):
        device = self._resolve_device()
        wide = build_wide_matrix(X)
        self.columns_ = wide.columns
        self.col_mean_ = wide.mean()
        self.global_mean_ = float(wide.to_numpy().mean())

        Xs, Ys, n_windows = make_windows(wide, self.lookback_weeks, self.horizon_weeks)
        self.n_windows_ = n_windows

        Xs_t = torch.tensor(Xs, device=device)
        Ys_t = torch.tensor(Ys, device=device)

        n_channels = wide.shape[1]
        self.model_ = DLinear(
            self.lookback_weeks, self.horizon_weeks, n_channels,
            kernel_size=self.kernel_size, individual=self.individual, normalize=self.normalize,
        ).to(device)

        horizon_dates = pd.date_range(wide.index[self.lookback_weeks], periods=n_windows + self.horizon_weeks - 1, freq="7D")
        holiday_flags = IS_HOLIDAY_BY_DATE.reindex(horizon_dates).fillna(False).to_numpy()
        holiday_w_np = np.stack([np.where(holiday_flags[i:i + self.horizon_weeks], 5.0, 1.0) for i in range(n_windows)])
        weight_t = torch.tensor(holiday_w_np, dtype=torch.float32, device=device).unsqueeze(-1) if self.use_sample_weight else None

        opt = torch.optim.Adam(self.model_.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        self.model_.train()
        n = Xs_t.shape[0]
        for epoch in range(self.epochs):
            perm = torch.randperm(n, device=device)
            epoch_loss_sum = 0.0
            for i in range(0, n, self.batch_size):
                idx = perm[i:i + self.batch_size]
                opt.zero_grad()
                pred = self.model_(Xs_t[idx])
                if weight_t is not None:
                    loss = (weight_t[idx] * (pred - Ys_t[idx]).abs()).mean()
                else:
                    loss = (pred - Ys_t[idx]).abs().mean()
                loss.backward()
                opt.step()
                epoch_loss_sum += loss.item() * len(idx)
            if wandb.run is not None:
                wandb.log({"epoch": epoch, "epoch_loss": epoch_loss_sum / n})

        self.model_.eval()
        with torch.no_grad():
            train_pred = self.model_(Xs_t).cpu().numpy().astype(np.float64)
        train_true = Ys.astype(np.float64)
        diff = np.abs(train_true - train_pred)
        holiday_w_bc = holiday_w_np[:, :, None]
        self.train_wmae_ = float(np.sum(holiday_w_bc * diff) / (np.sum(holiday_w_np) * diff.shape[-1]))

        self.last_window_ = wide.to_numpy(dtype=np.float32)[-self.lookback_weeks:]
        self.forecast_start_ = wide.index[-1] + pd.Timedelta(weeks=1)

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(X)
        return self

    def _forecast_grid(self):
        device = self._resolve_device()
        self.model_.eval()
        with torch.no_grad():
            x = torch.tensor(self.last_window_[None, :, :], device=device)
            pred = self.model_(x)[0].cpu().numpy()
        dates = pd.date_range(self.forecast_start_, periods=self.horizon_weeks, freq="7D")
        return pd.DataFrame(pred, index=dates, columns=self.columns_)

    def predict(self, X):
        grid = self._forecast_grid()
        long = grid.stack().rename("pred").reset_index()
        long.columns = ["Date", "_key", "pred"]

        lookup = X[["Store", "Dept", "Date"]].copy()
        lookup["_key"] = X["Store"].astype(str) + "|" + X["Dept"].astype(str)
        merged = lookup.merge(long, on=["_key", "Date"], how="left")
        preds = merged["pred"].to_numpy(copy=True)

        missing = np.isnan(preds)
        if missing.any():
            fallback = merged.loc[missing, "_key"].map(self.col_mean_)
            preds[missing] = fallback.fillna(self.global_mean_).to_numpy()

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X, preds)
        return preds

print("DLinear classes redeclared.")

DLinear classes redeclared.


In [7]:
dlinear_artifact = run.use_artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{DLINEAR_ARTIFACT_NAME}:latest", type="model")
dlinear_artifact_dir = dlinear_artifact.download()
print("Downloaded DLinear artifact version:", dlinear_artifact.version, "->", dlinear_artifact_dir)
print("DLinear artifact metadata:", dlinear_artifact.metadata)

with open(os.path.join(dlinear_artifact_dir, DLINEAR_PKL_FILENAME), "rb") as f:
    dlinear_pipeline = pickle.load(f)
print("Loaded DLinear pipeline:", type(dlinear_pipeline).__name__)

wandb:   1 of 1 files downloaded.  


Downloaded DLinear artifact version: v3 -> /Users/macbookpro/PycharmProjects/machine_learning_final_project/artifacts/dlinear_forecast_pipeline:v3
DLinear artifact metadata: {'epochs': 60, 'normalize': True, 'batch_size': 32, 'individual': False, 'kernel_size': 51, 'cv_wmae_full': 3124.404487886157, 'weight_decay': 0.1, 'learning_rate': 0.02, 'lookback_weeks': 52, 'deploy_train_wmae': 1608.154773362777, 'use_sample_weight': False, 'cv_train_wmae_full': 1187.7472156072429, 'cv_holiday_mae_full': 4304.176818768005, 'use_christmas_shift': False, 'cv_nonholiday_mae_full': 2626.073231686707, 'cv_wmae_christmas_shift': 3148.092644609039}
Loaded DLinear pipeline: DLinearForecastPipeline


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
import lightgbm as lgb

SEED = 42

MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]
FFILL_COLS = ["CPI", "Unemployment"]
MEDIAN_ONLY_COLS = ["Temperature", "Fuel_Price"]

class DataCleaner(BaseEstimator, TransformerMixin):

    def __init__(self, markdown_cols=None, ffill_cols=None, median_only_cols=None):
        self.markdown_cols = markdown_cols if markdown_cols is not None else MARKDOWN_COLS
        self.ffill_cols = ffill_cols if ffill_cols is not None else FFILL_COLS
        self.median_only_cols = median_only_cols if median_only_cols is not None else MEDIAN_ONLY_COLS

    def fit(self, X, y=None):
        X_sorted = X.sort_values(["Store", "Date"])
        filled = X_sorted.groupby("Store")[self.ffill_cols].ffill()
        self.last_known_ = filled.groupby(X_sorted["Store"]).last()
        self.medians_ = {c: X[c].median() for c in self.ffill_cols + self.median_only_cols}
        has_markdown = X[self.markdown_cols].notna().any(axis=1)
        self.markdown_cutoff_ = X.loc[has_markdown, "Date"].min() if has_markdown.any() else X["Date"].max()
        return self

    def transform(self, X):
        X = X.sort_values(["Store", "Date"]).copy()

        X[self.markdown_cols] = X[self.markdown_cols].fillna(0.0)
        X["markdown_recorded"] = (X["Date"] >= self.markdown_cutoff_).astype(int)

        for c in self.ffill_cols:
            X[c] = X.groupby("Store")[c].ffill()
            X[c] = X[c].fillna(X["Store"].map(self.last_known_[c]))
            X[c] = X[c].fillna(self.medians_[c])
        for c in self.median_only_cols:
            X[c] = X[c].fillna(self.medians_[c])

        return X.sort_index()

THANKSGIVING_DATES = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
SUPERBOWL_DATES = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
CHRISTMAS_DATES = pd.to_datetime(["2010-12-25", "2011-12-25", "2012-12-25", "2013-12-25"])
LAGS = [39, 52, 104]

def _signed_days_to_nearest(dates, anchors):
    arr = dates.values.astype("datetime64[D]")
    anc = np.asarray(anchors, dtype="datetime64[D]")
    diffs = (arr[:, None] - anc[None, :]).astype("timedelta64[D]").astype(int)
    idx = np.abs(diffs).argmin(axis=1)
    return diffs[np.arange(len(arr)), idx]

class TimeSeriesFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, lags=None):
        self.lags = lags if lags is not None else LAGS

    def fit(self, X, y=None):
        self.history_ = X[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
        self.store_dept_mean_ = X.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
        self.dept_type_mean_ = X.groupby(["Dept", "Type"])["Weekly_Sales"].mean()
        self.global_mean_ = X["Weekly_Sales"].mean()
        woy = X["Date"].dt.isocalendar().week.astype(int)
        self.woy_mean_ = X.assign(weekofyear=woy).groupby(["Store", "Dept", "weekofyear"])["Weekly_Sales"].mean()
        self.hist_start_ = X.groupby(["Store", "Dept"])["Date"].min()
        self.store_categories_ = sorted(X["Store"].unique())
        self.dept_categories_ = sorted(X["Dept"].unique())
        self.type_categories_ = sorted(X["Type"].unique())
        return self

    def transform(self, X):
        X = X.copy()
        X["month"] = X["Date"].dt.month
        X["weekofyear"] = X["Date"].dt.isocalendar().week.astype(int)
        X["days_to_thanksgiving"] = _signed_days_to_nearest(X["Date"], THANKSGIVING_DATES)
        X["days_to_christmas"] = _signed_days_to_nearest(X["Date"], CHRISTMAS_DATES)
        X["days_to_superbowl"] = _signed_days_to_nearest(X["Date"], SUPERBOWL_DATES)

        for lag in self.lags:
            lag_tbl = self.history_.rename(columns={"Weekly_Sales": f"lag_{lag}"}).copy()
            lag_tbl["Date"] = lag_tbl["Date"] + pd.Timedelta(weeks=lag)
            X = X.merge(lag_tbl, on=["Store", "Dept", "Date"], how="left")

        sd_mean_tbl = self.store_dept_mean_.rename("store_dept_mean").reset_index()
        X = X.merge(sd_mean_tbl, on=["Store", "Dept"], how="left")
        X["store_dept_mean"] = X["store_dept_mean"].fillna(self.global_mean_)

        dt_mean_tbl = self.dept_type_mean_.rename("dept_type_mean").reset_index()
        X = X.merge(dt_mean_tbl, on=["Dept", "Type"], how="left")
        X["dept_type_mean"] = X["dept_type_mean"].fillna(self.global_mean_)

        woy_mean_tbl = self.woy_mean_.rename("sd_woy_mean").reset_index()
        X = X.merge(woy_mean_tbl, on=["Store", "Dept", "weekofyear"], how="left")
        X["sd_woy_mean"] = X["sd_woy_mean"].fillna(X["store_dept_mean"])

        hist_start_tbl = self.hist_start_.rename("_hist_start").reset_index()
        X = X.merge(hist_start_tbl, on=["Store", "Dept"], how="left")
        X["hist_len"] = ((X["Date"] - X["_hist_start"]).dt.days / 7.0).clip(lower=0)
        X["hist_len"] = X["hist_len"].fillna(0.0)
        X = X.drop(columns=["_hist_start"])

        X["Store"] = pd.Categorical(X["Store"], categories=self.store_categories_)
        X["Dept"] = pd.Categorical(X["Dept"], categories=self.dept_categories_)
        X["Type"] = pd.Categorical(X["Type"], categories=self.type_categories_)

        return X

CORE_FEATURES = [
    "Store", "Dept", "Type", "Size",
    "month", "weekofyear", "days_to_thanksgiving", "days_to_christmas", "days_to_superbowl",
    "lag_39", "lag_52", "lag_104",
    "store_dept_mean", "dept_type_mean", "sd_woy_mean", "hist_len",
]
EXOGENOUS_FEATURES = [
    "IsHoliday", "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "markdown_recorded",
]
CATEGORICAL_FEATURES = ["Store", "Dept", "Type"]

CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

class LGBMForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, num_leaves=31, max_depth=-1, learning_rate=0.1, n_estimators=100,
                 min_child_samples=20, colsample_bytree=1.0, subsample=1.0, subsample_freq=0,
                 reg_alpha=0.0, reg_lambda=0.0, objective="regression",
                 use_exogenous=True, use_sample_weight=False,
                 use_christmas_shift=False, christmas_bulge_threshold=1.10,
                 early_stopping_weeks=8, early_stopping_rounds=50):
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.min_child_samples = min_child_samples
        self.colsample_bytree = colsample_bytree
        self.subsample = subsample
        self.subsample_freq = subsample_freq
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.objective = objective
        self.use_exogenous = use_exogenous
        self.use_sample_weight = use_sample_weight
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold
        self.early_stopping_weeks = early_stopping_weeks
        self.early_stopping_rounds = early_stopping_rounds

    def _feature_list(self):
        return CORE_FEATURES + (EXOGENOUS_FEATURES if self.use_exogenous else [])

    def fit(self, X, y=None):
        self.cleaner_ = DataCleaner().fit(X)
        X_clean = self.cleaner_.transform(X)

        cutoff = X_clean["Date"].max() - pd.Timedelta(weeks=self.early_stopping_weeks)
        fit_part = X_clean[X_clean["Date"] <= cutoff]
        es_part = X_clean[X_clean["Date"] > cutoff]

        fit_engineer = TimeSeriesFeatureEngineer().fit(fit_part)
        fit_feat = fit_engineer.transform(fit_part)
        es_feat = fit_engineer.transform(es_part) if len(es_part) else None

        feats = self._feature_list()
        cat_cols = [c for c in CATEGORICAL_FEATURES if c in feats]
        weight = np.where(fit_feat["IsHoliday"], 5.0, 1.0) if self.use_sample_weight else None

        self.model_ = lgb.LGBMRegressor(
            num_leaves=self.num_leaves, max_depth=self.max_depth, learning_rate=self.learning_rate,
            n_estimators=self.n_estimators, min_child_samples=self.min_child_samples,
            colsample_bytree=self.colsample_bytree, subsample=self.subsample,
            subsample_freq=self.subsample_freq, reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            objective=self.objective, verbosity=-1, random_state=SEED,
        )
        fit_kwargs = dict(categorical_feature=cat_cols)
        if es_feat is not None and len(es_feat):
            fit_kwargs["eval_set"] = [(es_feat[feats], es_feat["Weekly_Sales"])]
            fit_kwargs["callbacks"] = [lgb.early_stopping(self.early_stopping_rounds, verbose=False)]
        self.model_.fit(fit_feat[feats], fit_feat["Weekly_Sales"], sample_weight=weight, **fit_kwargs)

        self.engineer_ = TimeSeriesFeatureEngineer().fit(X_clean)

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(X_clean)
        return self

    def predict(self, X):
        X_clean = self.cleaner_.transform(X)
        X_feat = self.engineer_.transform(X_clean)
        feats = self._feature_list()
        preds = self.model_.predict(X_feat[feats])

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X_feat, preds)
        return preds

print("LightGBM classes redeclared.")

LightGBM classes redeclared.


In [9]:
lightgbm_artifact = run.use_artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{LIGHTGBM_ARTIFACT_NAME}:latest", type="model")
lightgbm_artifact_dir = lightgbm_artifact.download()
print("Downloaded LightGBM artifact version:", lightgbm_artifact.version, "->", lightgbm_artifact_dir)
print("LightGBM artifact metadata:", lightgbm_artifact.metadata)

with open(os.path.join(lightgbm_artifact_dir, LIGHTGBM_PKL_FILENAME), "rb") as f:
    lightgbm_pipeline = pickle.load(f)
print("Loaded LightGBM pipeline:", type(lightgbm_pipeline).__name__)

wandb:   1 of 1 files downloaded.  


Downloaded LightGBM artifact version: v1 -> /Users/macbookpro/PycharmProjects/machine_learning_final_project/artifacts/lightgbm_forecast_pipeline:v1
LightGBM artifact metadata: {'max_depth': -1, 'objective': 'regression', 'reg_alpha': 1.0, 'subsample': 1.0, 'num_leaves': 15, 'reg_lambda': 0.0, 'cv_wmae_full': 2113.376560087793, 'n_estimators': 3000, 'learning_rate': 0.05, 'use_exogenous': True, 'subsample_freq': 0, 'colsample_bytree': 0.6, 'min_child_samples': 20, 'use_sample_weight': False, 'use_christmas_shift': True}
Loaded LightGBM pipeline: LGBMForecastPipeline


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from prophet import Prophet

MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]
MEDIAN_FILL_COLS = ["CPI", "Unemployment", "Temperature", "Fuel_Price"]

class DataCleaner(BaseEstimator, TransformerMixin):

    def __init__(self, markdown_cols=None, median_cols=None):
        self.markdown_cols = markdown_cols if markdown_cols is not None else MARKDOWN_COLS
        self.median_cols = median_cols if median_cols is not None else MEDIAN_FILL_COLS

    def fit(self, X, y=None):
        self.medians_ = {c: X[c].median() for c in self.median_cols}
        return self

    def transform(self, X):
        X = X.copy()
        X[self.markdown_cols] = X[self.markdown_cols].fillna(0.0)
        for c in self.median_cols:
            X[c] = X[c].fillna(self.medians_[c])
        return X

def _build_holidays_table(lower_window=-3, upper_window=1):
    raw = {
        "Super Bowl":   ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
        "Labor Day":    ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
        "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
        "Christmas":    ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
    }
    rows = []
    for name, dates in raw.items():
        for d in dates:
            rows.append({"holiday": name, "ds": pd.Timestamp(d),
                         "lower_window": lower_window, "upper_window": upper_window})
    return pd.DataFrame(rows)

class HolidayFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, lower_window=-3, upper_window=1):
        self.lower_window = lower_window
        self.upper_window = upper_window

    def fit(self, X=None, y=None):
        self.holidays_df_ = _build_holidays_table(self.lower_window, self.upper_window)
        return self

    def transform(self, X):
        return X

CANDIDATE_REGRESSOR_COLS = ["IsHoliday"] + MARKDOWN_COLS + ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
STRUCTURALLY_EXCLUDED_COLS = list(MARKDOWN_COLS)

class RegressorSelector(BaseEstimator, TransformerMixin):

    def __init__(self, candidate_cols=None, structurally_excluded=None):
        self.candidate_cols = candidate_cols if candidate_cols is not None else CANDIDATE_REGRESSOR_COLS
        self.structurally_excluded = structurally_excluded if structurally_excluded is not None else STRUCTURALLY_EXCLUDED_COLS

    def fit(self, X, y=None):
        target = y if y is not None else X["Weekly_Sales"]
        corr_frame = X[self.candidate_cols].copy()
        if "IsHoliday" in corr_frame.columns:
            corr_frame["IsHoliday"] = corr_frame["IsHoliday"].astype(int)
        self.correlations_ = corr_frame.corrwith(target).abs().sort_values(ascending=False)
        self.selected_regressors_ = [c for c in self.candidate_cols if c not in self.structurally_excluded]
        return self

    def transform(self, X):
        key_cols = [c for c in ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"] if c in X.columns]
        keep = key_cols + [c for c in self.selected_regressors_ if c not in key_cols]
        return X[keep]

MIN_HISTORY_WEEKS = 52

class SeasonalNaiveFallback:

    def __init__(self, min_history_weeks=MIN_HISTORY_WEEKS):
        self.min_history_weeks = min_history_weeks

    def is_needed(self, history_df):
        if len(history_df) == 0:
            return True
        n_hist = history_df["Date"].nunique()
        series_mean = history_df["Weekly_Sales"].mean()
        return n_hist < self.min_history_weeks or history_df["Weekly_Sales"].std() == 0 or np.isnan(series_mean)

    def predict(self, history_df, future_df, global_fallback_mean):
        series_mean = history_df["Weekly_Sales"].mean() if len(history_df) else global_fallback_mean
        fallback_value = series_mean if not np.isnan(series_mean) else global_fallback_mean

        lag = history_df[["Date", "Weekly_Sales"]].rename(columns={"Date": "lag_date", "Weekly_Sales": "snaive"})
        fd = future_df.copy()
        fd["lag_date"] = fd["Date"] - pd.Timedelta(weeks=52)
        fd = fd.merge(lag, on="lag_date", how="left")
        return fd["snaive"].fillna(fallback_value).values

class ProphetSeriesModel(BaseEstimator, RegressorMixin):

    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 holidays_prior_scale=10.0, seasonality_mode="additive",
                 changepoint_range=0.8, n_changepoints=25, yearly_seasonality_order=10,
                 use_regressors=True, regressor_cols=None, holidays_df=None,
                 min_history_weeks=MIN_HISTORY_WEEKS):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.holidays_prior_scale = holidays_prior_scale
        self.seasonality_mode = seasonality_mode
        self.changepoint_range = changepoint_range
        self.n_changepoints = n_changepoints
        self.yearly_seasonality_order = yearly_seasonality_order
        self.use_regressors = use_regressors
        self.regressor_cols = regressor_cols or []
        self.holidays_df = holidays_df
        self.min_history_weeks = min_history_weeks

    def fit(self, X, y=None):
        self.fallback_ = SeasonalNaiveFallback(self.min_history_weeks)
        self.history_ = X
        self.needs_fallback_ = self.fallback_.is_needed(X)

        if not self.needs_fallback_:
            prophet_df = X.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"] + self.regressor_cols]
            self.model_ = Prophet(
                holidays=self.holidays_df,
                changepoint_prior_scale=self.changepoint_prior_scale,
                seasonality_prior_scale=self.seasonality_prior_scale,
                holidays_prior_scale=self.holidays_prior_scale,
                seasonality_mode=self.seasonality_mode,
                changepoint_range=self.changepoint_range,
                n_changepoints=self.n_changepoints,
                yearly_seasonality=self.yearly_seasonality_order,
                weekly_seasonality=False,
                daily_seasonality=False,
            )
            if self.use_regressors:
                for col in self.regressor_cols:
                    self.model_.add_regressor(col)
            self.model_.fit(prophet_df)
        return self

    def predict(self, X, global_fallback_mean=0.0):
        if self.needs_fallback_:
            return self.fallback_.predict(self.history_, X, global_fallback_mean)

        cols = ["ds"] + (self.regressor_cols if self.use_regressors else [])
        future = X.rename(columns={"Date": "ds"})[cols]
        forecast = self.model_.predict(future)
        return forecast["yhat"].values

CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

class ProphetForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 holidays_prior_scale=10.0, seasonality_mode="additive",
                 changepoint_range=0.8, n_changepoints=25, yearly_seasonality_order=10,
                 use_regressors=True, min_history_weeks=MIN_HISTORY_WEEKS,
                 holiday_window=(-3, 1), use_christmas_shift=True, christmas_bulge_threshold=1.10):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.holidays_prior_scale = holidays_prior_scale
        self.seasonality_mode = seasonality_mode
        self.changepoint_range = changepoint_range
        self.n_changepoints = n_changepoints
        self.yearly_seasonality_order = yearly_seasonality_order
        self.use_regressors = use_regressors
        self.min_history_weeks = min_history_weeks
        self.holiday_window = holiday_window
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold

    def fit(self, X, y=None):
        self.cleaner_ = DataCleaner().fit(X)
        X_clean = self.cleaner_.transform(X)

        self.engineer_ = HolidayFeatureEngineer(*self.holiday_window).fit()

        self.selector_ = RegressorSelector().fit(X_clean)
        self.train_ = self.selector_.transform(X_clean)

        self.global_fallback_mean_ = self.train_["Weekly_Sales"].mean()

        self.series_models_ = {}
        for (store, dept), history in self.train_.groupby(["Store", "Dept"]):
            series_model = self._make_series_model()
            series_model.fit(history.sort_values("Date"))
            self.series_models_[(store, dept)] = series_model

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(self.train_)
        return self

    def _make_series_model(self):
        return ProphetSeriesModel(
            changepoint_prior_scale=self.changepoint_prior_scale,
            seasonality_prior_scale=self.seasonality_prior_scale,
            holidays_prior_scale=self.holidays_prior_scale,
            seasonality_mode=self.seasonality_mode,
            changepoint_range=self.changepoint_range,
            n_changepoints=self.n_changepoints,
            yearly_seasonality_order=self.yearly_seasonality_order,
            use_regressors=self.use_regressors,
            regressor_cols=self.selector_.selected_regressors_,
            holidays_df=self.engineer_.holidays_df_,
            min_history_weeks=self.min_history_weeks,
        )

    def predict(self, X):
        X_clean = self.cleaner_.transform(X)
        X_sel = self.selector_.transform(X_clean)

        unseen_fallback_ = SeasonalNaiveFallback(self.min_history_weeks)

        preds = np.zeros(len(X_sel))
        for (store, dept), future in X_sel.groupby(["Store", "Dept"]):
            future_sorted = future.sort_values("Date")
            series_model = self.series_models_.get((store, dept))
            if series_model is not None:
                preds[future_sorted.index.values] = series_model.predict(future_sorted, self.global_fallback_mean_)
            else:
                history = self.train_[(self.train_.Store == store) & (self.train_.Dept == dept)].sort_values("Date")
                preds[future_sorted.index.values] = unseen_fallback_.predict(
                    history, future_sorted, self.global_fallback_mean_
                )

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X_sel, preds)

        return preds

print("Prophet classes redeclared.")

Importing plotly failed. Interactive plots will not work.


Prophet classes redeclared.


In [11]:
prophet_artifact = run.use_artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{PROPHET_ARTIFACT_NAME}:latest", type="model")
prophet_artifact_dir = prophet_artifact.download()
print("Downloaded Prophet artifact version:", prophet_artifact.version, "->", prophet_artifact_dir)
print("Prophet artifact metadata:", prophet_artifact.metadata)

with open(os.path.join(prophet_artifact_dir, PROPHET_PKL_FILENAME), "rb") as f:
    prophet_pipeline = pickle.load(f)
print("Loaded Prophet pipeline:", type(prophet_pipeline).__name__)

wandb: Downloading large artifact 'prophet_forecast_pipeline:latest', 137.23MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.3 (478.3MB/s)


Downloaded Prophet artifact version: v3 -> /Users/macbookpro/PycharmProjects/machine_learning_final_project/artifacts/prophet_forecast_pipeline:v3
Prophet artifact metadata: {'cv_wmae_full': 2138.0, 'n_changepoints': 15, 'use_regressors': False, 'seasonality_mode': 'additive', 'changepoint_range': 0.95, 'use_christmas_shift': True, 'holidays_prior_scale': 1.0, 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 0.1, 'yearly_seasonality_order': 20}
Loaded Prophet pipeline: ProphetForecastPipeline


In [12]:
ids = (
    df_test["Store"].astype(str) + "_" +
    df_test["Dept"].astype(str) + "_" +
    df_test["Date"].dt.strftime("%Y-%m-%d")
)

preds_by_model = {}

t0 = time.time()
preds_by_model["DLinear"] = np.asarray(dlinear_pipeline.predict(df_test), dtype=float)
print(f"DLinear predicted {len(preds_by_model['DLinear'])} rows in {time.time() - t0:.1f}s")

t0 = time.time()
preds_by_model["LightGBM"] = np.asarray(lightgbm_pipeline.predict(df_test), dtype=float)
print(f"LightGBM predicted {len(preds_by_model['LightGBM'])} rows in {time.time() - t0:.1f}s")

t0 = time.time()
preds_by_model["Prophet"] = np.asarray(prophet_pipeline.predict(df_test), dtype=float)
print(f"Prophet predicted {len(preds_by_model['Prophet'])} rows in {(time.time() - t0) / 60:.1f} min")

for name, preds in preds_by_model.items():
    n_nan = int(np.isnan(preds).sum())
    assert n_nan == 0, f"{name} predict() produced {n_nan} NaNs -- investigate before blending"
    assert len(preds) == len(df_test), f"{name} predict() returned {len(preds)} rows, expected {len(df_test)}"


DLinear predicted 115064 rows in 0.1s
LightGBM predicted 115064 rows in 0.6s
Prophet predicted 115064 rows in 0.9 min


In [13]:
pred_df = pd.DataFrame(preds_by_model)
corr_matrix = pred_df.corr()
print("Pairwise prediction correlation:")
print(corr_matrix.round(4))

blend_weights = {name: 1.0 / len(preds_by_model) for name in preds_by_model}
blended_preds = sum(blend_weights[name] * preds for name, preds in preds_by_model.items())
print()
print("Blend weights (equal):", blend_weights)
print(f"Blended {len(blended_preds)} predictions")

Pairwise prediction correlation:
          DLinear  LightGBM  Prophet
DLinear    1.0000    0.9929   0.9876
LightGBM   0.9929    1.0000   0.9875
Prophet    0.9876    0.9875   1.0000

Blend weights (equal): {'DLinear': 0.3333333333333333, 'LightGBM': 0.3333333333333333, 'Prophet': 0.3333333333333333}
Blended 115064 predictions


In [14]:
submission = pd.DataFrame({
    "Id": ids,
    "Weekly_Sales": blended_preds,
})

assert len(submission) == len(df_test)
assert submission["Id"].is_unique

submission.to_csv(SUBMISSION_FILENAME, index=False)
print(f"Saved {SUBMISSION_FILENAME} ({len(submission)} rows)")

submission_artifact = wandb.Artifact(
    "ensemble_dlinear_lightgbm_prophet_submission", type="submission",
    metadata={
        "n_rows": len(submission),
        "blend_weights": blend_weights,
        "prediction_correlation": corr_matrix.to_dict(),
        "dlinear_metadata": dlinear_artifact.metadata,
        "lightgbm_metadata": lightgbm_artifact.metadata,
        "prophet_metadata": prophet_artifact.metadata,
    },
)
submission_artifact.add_file(SUBMISSION_FILENAME)

submission.head()

Saved submission_Ensemble_DLinear_LightGBM_Prophet.csv (115064 rows)


,Id,Weekly_Sales
0,1_1_2012-11-02,36327.903102
1,1_1_2012-11-09,21587.524835
2,1_1_2012-11-16,18773.229448
3,1_1_2012-11-23,20100.660779
4,1_1_2012-11-30,24421.516206
